# Synthetic GPU benchmark (V100 / any CUDA GPU)

Small transformer-style stack: forward + backward + AdamW. **Not** Nemotron or vLLM — only a **repeatable** throughput check.

Measured results and interpretation: **`docs/GPU_SYNTH_BENCHMARK.md`**.

**Requirements:** PyTorch with CUDA. No Hugging Face token, no env files.

**Multi-GPU (DDP):** Jupyter is usually one process per notebook. For `4× V100` numbers, run from a shell at repo root:

```bash
torchrun --nproc_per_node=4 scripts/benchmark_gpu.py \\
  --batch 4 --seq 256 --hidden 3072 --layers 10 \\
  --warmup 5 --steps 25 --dtype fp16
```


In [ ]:
# --- knobs (keep identical when comparing hardware) ---
BATCH = 4
SEQ = 256
HIDDEN = 3072
LAYERS = 10
WARMUP = 5
STEPS = 25
DTYPE = "fp16"  # "fp16" | "bf16"
LR = 1e-4

import os
import time

import torch
import torch.nn as nn

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required (use a GPU kernel).")

device = torch.device("cuda", 0)
dtype = torch.float16 if DTYPE == "fp16" else torch.bfloat16
if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
    print("bf16 not supported; using fp16")
    dtype = torch.float16

print("GPU:", torch.cuda.get_device_name(device))
print("dtype:", dtype)
print("config:", dict(BATCH=BATCH, SEQ=SEQ, HIDDEN=HIDDEN, LAYERS=LAYERS))

In [ ]:
class Block(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 4 * hidden),
            nn.GELU(),
            nn.Linear(4 * hidden, hidden),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(x)


class TinyTrainModel(nn.Module):
    def __init__(self, hidden: int, layers: int):
        super().__init__()
        self.blocks = nn.ModuleList([Block(hidden) for _ in range(layers)])
        self.head = nn.Linear(hidden, hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for b in self.blocks:
            x = b(x)
        return self.head(x)


model = TinyTrainModel(hidden=HIDDEN, layers=LAYERS).to(device=device, dtype=dtype)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

x = torch.randn(BATCH, SEQ, HIDDEN, device=device, dtype=dtype)
target = torch.randn(BATCH, SEQ, HIDDEN, device=device, dtype=dtype)


def step() -> float:
    t0 = time.perf_counter()
    opt.zero_grad(set_to_none=True)
    out = model(x)
    loss = ((out - target) ** 2).mean()
    loss.backward()
    opt.step()
    torch.cuda.synchronize(device)
    return time.perf_counter() - t0


for _ in range(WARMUP):
    step()

times = [step() for _ in range(STEPS)]
avg_s = sum(times) / len(times)
world_size = int(os.environ.get("WORLD_SIZE", "1"))
global_tokens_per_step = BATCH * SEQ * world_size
toks_per_sec = global_tokens_per_step / avg_s
approx_flops_per_token = 2.0 * LAYERS * HIDDEN * (4 * HIDDEN) * 2.0
approx_tflops = (toks_per_sec * approx_flops_per_token) / 1e12

print()
print("=== benchmark_result ===")
print(f"gpu={torch.cuda.get_device_name(device)}")
print(f"world_size={world_size}")
print(f"dtype={str(dtype).replace('torch.', '')}")
print(f"batch={BATCH} seq={SEQ} hidden={HIDDEN} layers={LAYERS}")
print(f"avg_step_s={avg_s:.6f}")
print(f"tokens_per_sec={toks_per_sec:.2f}")
print(f"approx_tflops={approx_tflops:.2f}")